# 15. VAAS — Patch Scores, Feature Extraction, and Batch Inference

This notebook demonstrates three APIs that are especially useful for research workflows:

- `extract_patch_scores`
- `extract_features`
- `batch`

These APIs help expose intermediate outputs for localisation, feature analysis, and multi-image inference.

## 1. Install dependencies

In [ ]:
!pip install -q vaas torch torchvision

## 2. Imports

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image
from IPython.display import display

from vaas.inference.pipeline import VAASPipeline

## 3. Load example images

We use two public example images from the VAAS repository.

In [ ]:
image_urls = {
    "example_1": "https://raw.githubusercontent.com/OBA-Research/VAAS/main/examples/images/COCO_DF_C110B00000_00539519.jpg",
    "example_2": "https://raw.githubusercontent.com/OBA-Research/VAAS/main/examples/images/COCO_DF_S000B00000_00120651.jpg",
}

images = {
    name: Image.open(BytesIO(requests.get(url).content)).convert("RGB")
    for name, url in image_urls.items()
}

display(images["example_1"])
display(images["example_2"])

## 4. Load pipeline

In [ ]:
pipeline = VAASPipeline.from_pretrained(
    repo_id="OBA-Research/vaas",
    model_variant="v2-large-df2023",
    device="cpu",   # change to "cuda" if GPU is available
    alpha=0.5,
)

## 5. Extract patch scores

`extract_patch_scores` returns the dense patch-level anomaly map directly.

In [ ]:
patch_scores = pipeline.extract_patch_scores(images["example_1"])

print("Patch score shape:", patch_scores.shape)
print("Min:", float(patch_scores.min()))
print("Max:", float(patch_scores.max()))
print("Mean:", float(patch_scores.mean()))

## 6. Visualise patch scores

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(patch_scores, cmap="jet")
plt.colorbar()
plt.title("Patch-level anomaly scores")
plt.axis("off")
plt.show()

## 7. Overlay patch scores on image

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(images["example_1"])
plt.imshow(patch_scores, cmap="jet", alpha=0.4)
plt.title("Patch scores overlay")
plt.axis("off")
plt.show()

## 8. Extract features

`extract_features` exposes three feature groups:

- `fx_cls_embedding`
- `fx_patch_tokens`
- `px_cross_attention_tokens`

In [ ]:
features = pipeline.extract_features(images["example_1"])

print(features.keys())
print("fx_cls_embedding:", features["fx_cls_embedding"].shape)
print("fx_patch_tokens:", features["fx_patch_tokens"].shape)
print("px_cross_attention_tokens:", features["px_cross_attention_tokens"].shape)

## 9. Inspect feature arrays

In [ ]:
fx_cls = features["fx_cls_embedding"]
fx_patches = features["fx_patch_tokens"]
px_tokens = features["px_cross_attention_tokens"]

print("fx_cls first values:", fx_cls.flatten()[:10])
print("fx_patch_tokens first values:", fx_patches.flatten()[:10])
print("px_cross_attention_tokens first values:", px_tokens.flatten()[:10])

## 10. Example research uses for extracted features

These outputs can be used for:

- feature similarity analysis
- clustering
- retrieval experiments
- anomaly-aware downstream models
- probing cross-attention behaviour

## 11. Batch inference

`batch` runs the standard pipeline over multiple images and returns a list of outputs.

In [ ]:
batch_results = pipeline.batch(
    [
        images["example_1"],
        images["example_2"],
    ]
)

print("Number of batch results:", len(batch_results))
print(batch_results[0].keys())

## 12. Summarise batch outputs

In [ ]:
for i, out in enumerate(batch_results, start=1):
    print(f"Image {i}")
    print("  S_F:", out["S_F"])
    print("  S_P:", out["S_P"])
    print("  S_H:", out["S_H"])
    print("  anomaly_map shape:", out["anomaly_map"].shape)

## 13. Rank batch inputs by anomaly score

In [ ]:
ranked = sorted(
    [
        ("example_1", batch_results[0]["S_H"]),
        ("example_2", batch_results[1]["S_H"]),
    ],
    key=lambda x: x[1],
    reverse=True,
)

ranked

## 14. Save research outputs

A simple example of storing extracted arrays for downstream use.

In [ ]:
np.save("example_1_patch_scores.npy", patch_scores)
np.save("example_1_fx_cls_embedding.npy", features["fx_cls_embedding"])
np.save("example_1_fx_patch_tokens.npy", features["fx_patch_tokens"])
np.save("example_1_px_cross_attention_tokens.npy", features["px_cross_attention_tokens"])

print("Saved patch scores and extracted features")

## 15. Summary

In this notebook you:

- extracted patch-level anomaly scores
- visualised patch score maps directly
- extracted Fx and Px intermediate features
- ran batch inference over multiple images
- ranked images by anomaly score

This notebook is useful for researcher-facing workflows that go beyond standard visualisation.

## 16. Next notebook
Next notebook: [17_structural_vs_ai_generated_examples.ipynb](https://colab.research.google.com/drive/1ffraWccft-MjR0jzuf-9VkvRcTFSjrFg?usp=sharing)